# Project 3: Eliminating Child Care Deserts in New York State through Optimization

## Problem A: Budgeting
Determining the minimum amount of funding (in total) needed to meet their target for each area, categorized by zip code

In [123]:
import numpy as numpy
import pandas as pd

In [124]:
df = pd.read_csv("cleaned_data.csv")
df

,zipcode,r_a,e_a,p_t_a,p_u_a,high_demand,n_t_a,n_u_a,is_desert,is_under5_desert
0,10001,0.595097,102878.033603,4.0,0,False,609.0,0.0,False,False
1,10002,0.520662,59604.041165,2093.2,744,True,4729.0,18.0,False,True
2,10003,0.497244,114273.049645,7106.8,2142,False,1995.0,0.0,True,True
3,10004,0.506661,132004.310345,3045.8,1440,False,263.0,0.0,True,True
4,10005,0.665833,121437.713311,711.6,433,True,39.0,0.0,True,True
...,...,...,...,...,...,...,...,...,...,...
1370,14767,0.322296,54623.287671,31.6,0,True,16.0,0.0,False,False
1371,14770,0.446676,55523.255814,28.0,0,True,70.0,15.0,False,False
1372,14772,0.410719,57164.634146,891.6,375,True,108.0,32.0,True,True
1373,14805,0.679739,59375.000000,141.2,20,True,8.0,0.0,True,True


#### Further Clean Facilities Data

In [125]:
facilities = pd.read_csv("./data/child_care_regulated.csv")
facilities["under_5_capacity"] = (facilities["infant_capacity"].fillna(0) + facilities["toddler_capacity"].fillna(0) + facilities["preschool_capacity"].fillna(0))
facilities

,facility_id,program_type,facility_status,facility_name,city,zip_code,school_district_name,infant_capacity,toddler_capacity,preschool_capacity,school_age_capacity,children_capacity,total_capacity,latitude,longitude,under_5_capacity
0,2416,FDC,Registration,"Bohrer, Barbara",Clinton,13323,Clinton,0,0,0,2,6,8,NaN,NaN,0
1,5555,FDC,Registration,"Matey, Sally",Jamestown,14701,Jamestown,0,0,0,2,6,8,NaN,NaN,0
2,9066,FDC,Registration,"Copeland, Denise",Wappingers Falls,12590,Wappingers,0,0,0,2,6,8,NaN,NaN,0
3,40163,DCC,License,"Head Start of Rockland, Inc.",Nyack,10960,Nyack,0,10,110,0,0,120,41.089425,-73.920413,120
4,41016,SACC,Registration,"School's Out, Inc.",Glenmont,12077,Bethlehem,0,0,0,75,0,75,42.607043,-73.788606,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15599,892735,GFDC,License,LITTLE LILIES GROUP FAMILY DAYCARE LLC.,Bronx,10462,Bronx 11,0,0,0,4,12,16,40.854317,-73.864996,0
15600,897263,GFDC,License,"Cummings, Darlene",Brooklyn,11205,Brooklyn 13,0,0,0,0,10,10,40.696940,-73.977121,0
15601,901966,GFDC,License,"Pascal Genao, Angela",Yonkers,10705,Yonkers,0,0,0,4,12,16,40.910754,-73.893528,0
15602,892455,GFDC,License,"Warnakulasuriya, Sajeeka",Staten Island,10303,Richmond 31,0,0,0,4,12,16,40.628144,-74.156228,0


In [126]:
facilities_dict = {}
for _,row in facilities.iterrows():
    zipcode = row["zip_code"]
    facility_info = (int(row["facility_id"]), int(row["total_capacity"]), int(row["under_5_capacity"]))
    if zipcode not in facilities_dict:
        facilities_dict[zipcode] = []
    facilities_dict[zipcode].append(facility_info)
facilities_dict

{13323: [(2416, 8, 0),
  (251944, 18, 18),
  (399551, 57, 57),
  (545765, 8, 0),
  (695907, 106, 56),
  (825128, 60, 0),
  (891527, 31, 31)],
 14701: [(5555, 8, 0),
  (818632, 16, 0),
  (177617, 40, 0),
  (609377, 64, 64),
  (264842, 8, 0),
  (769289, 40, 0),
  (39263, 73, 73),
  (688997, 226, 226),
  (869163, 70, 0),
  (668933, 8, 0),
  (19615, 8, 0),
  (584466, 16, 0),
  (862452, 8, 0),
  (174046, 70, 0),
  (147284, 45, 0),
  (200823, 70, 0),
  (540688, 62, 62),
  (799842, 8, 0),
  (385424, 16, 0),
  (16446, 8, 0),
  (243357, 73, 0),
  (801487, 28, 28),
  (33710, 16, 0),
  (358757, 16, 16),
  (39262, 106, 106),
  (841580, 8, 0),
  (864675, 8, 0),
  (701340, 8, 0),
  (914698, 8, 0),
  (911729, 8, 0),
  (921144, 8, 0),
  (910516, 71, 0),
  (896110, 8, 0)],
 12590: [(9066, 8, 0),
  (100010, 82, 82),
  (20973, 7, 0),
  (625561, 16, 0),
  (192440, 14, 0),
  (773085, 40, 0),
  (347661, 98, 98),
  (614527, 16, 0),
  (832206, 8, 8),
  (501166, 8, 0),
  (741408, 40, 0),
  (712601, 8, 0),
  (4

### Objective
The objective of Problem A is to determine the minimum amount of funding (in total) needed to meet the NYS government's target for each area, categorized by zip code.

The government plans to either:
1. build new child care facilities
2. expand exisiting ones

For this problem, we decided to use the PuLP library.
We referred to this page to learn more about the PuLP library.
https://www.geeksforgeeks.org/python-linear-programming-in-pulp/

#### Create a LP Minimization Problem

In [127]:
# pip install pulp
import pulp as p
model = p.LpProblem("Child_Care_Desert_Elimination", p.LpMinimize)

#### Define Costs
1. Small: Facility Size = 100, # of Slots (Ages 0-5) = 50, Cost of New Facility = 65000
2. Medium: Facility Size = 200, # of Slots (Ages 0-5) = 100, Cost of New Facility = 95000
3. Large: Facility Size = 400, # of Slots (Ages 0-5) = 200, Cost of New Facility = 115,000

In [128]:
new_facility_total_slots = {1: 100, 2: 200, 3: 400}
new_facility_under5_slots = {1: 50, 2: 100, 3: 200}
new_facility_cost = {1: 65000, 2: 95000, 3: 115000}

#### Create Decision Variables
- y_a_i : the number of new facility of type i to build in area a
- x_a_j : the ratio of extended slots over the current slots in facility j in area a
- m_t_a_j : the total number of extended slots for children in facility j in area a
- m_u_a_j : the number of extended slots for children ages 0-5 in facility j in area a
- b_a_j: whether the facility j in area a is expanded (=1 if expanded and 0 otherwise)

In [129]:
zipcodes = df["zipcode"].tolist()

y = {(a, i): p.LpVariable(f"y_{a}_{i}", lowBound=0, cat=p.LpInteger)
     for a in zipcodes for i in [1, 2, 3]}

In [130]:
x, m_t, m_u = {}, {}, {}
for a in zipcodes:
    for (j, nt, nu) in facilities.get(a, []):
        x[(a, j)] = p.LpVariable(f"x_{a}_{j}", lowBound=0, upBound=0.2, cat=p.LpContinuous)
        m_t[(a, j)] = p.LpVariable(f"m_t_{a}_{j}", lowBound=0, cat=p.LpInteger)
        m_u[(a, j)] = p.LpVariable(f"m_u_{a}_{j}", lowBound=0, cat=p.LpInteger)


#### Create Objective Function

In [131]:
model += (
    p.lpSum((20000 + 200 * nt) * x[(a, j)] + 100 * m_u[(a, j)]
          for a in zipcodes for (j, nt, _) in facilities.get(a, [])) +
    p.lpSum(y[(a, i)] * new_facility_cost[i]
          for a in zipcodes for i in [1, 2, 3])
)

#### Create Constraints

In [132]:
for idx, row in df.iterrows():
    a = row["zipcode"]
    existing_total = row["n_t_a"]
    existing_under5 = row["n_u_a"]
    p_t = row["p_t_a"]
    p_u = row["p_u_a"]
    threshold = 0.5 if row["high_demand"] else (1/3)

    # Total slot coverage
    model += (
        existing_total +
        p.lpSum(m_t[(a, j)] for (j, _, _) in facilities.get(a, [])) +
        p.lpSum(new_facility_total_slots[i] * y[(a, i)] for i in [1, 2, 3])
        >= threshold * p_t
    )

    # Under-5 slot coverage
    model += (
        existing_under5 +
        p.lpSum(m_u[(a, j)] for (j, _, _) in facilities.get(a, [])) +
        p.lpSum(new_facility_under5_slots[i] * y[(a, i)] for i in [1, 2, 3])
        >= (2/3) * p_u
    )


In [133]:
for a in zipcodes:
    for (j, nt, _) in facilities.get(a, []):
        model += m_t[(a, j)] == x[(a, j)] * nt
        model += m_u[(a, j)] <= m_t[(a, j)]
        if nt >= 500:
            model += x[(a, j)] == 0

In [140]:
from pulp import GLPK_CMD
model.solve(GLPK_CMD(msg=True))
print(f"Status: {model.status}, Objective Cost: ${value(model.objective):,.2f}")

GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --cpxlp /var/folders/rg/vtjd9wkj6cq1lck908ywby500000gn/T/240f3f8ea55a469188083c1c802f90e6-pulp.lp
 -o /var/folders/rg/vtjd9wkj6cq1lck908ywby500000gn/T/240f3f8ea55a469188083c1c802f90e6-pulp.sol
Reading problem data from '/var/folders/rg/vtjd9wkj6cq1lck908ywby500000gn/T/240f3f8ea55a469188083c1c802f90e6-pulp.lp'...
2750 rows, 4125 columns, 8250 non-zeros
4125 integer variables, none of which are binary
12038 lines were read
GLPK Integer Optimizer 5.0
2750 rows, 4125 columns, 8250 non-zeros
4125 integer variables, none of which are binary
Preprocessing...
1810 rows, 2979 columns, 5430 non-zeros
2979 integer variables, none of which are binary
Scaling...
 A: min|aij| =  5.000e+01  max|aij| =  4.000e+02  ratio =  8.000e+00
GM: min|aij| =  1.000e+00  max|aij| =  1.000e+00  ratio =  1.000e+00
EQ: min|aij| =  1.000e+00  max|aij| =  1.000e+00  ratio =  1.000e+00
2N: min|aij| =  7.812e-01  max|aij| =  7.812e-01  ratio =  

KeyboardInterrupt: 

SyntaxError: invalid syntax (224044529.py, line 1)